# Perseus 4 Static Site Generator

This notebook takes a Greek TEI XML text and its English translation and generates
a set of static HTML pages that replicate the look and feel of Perseus 4.0 (the "Hopper").

**Features replicated:**
- Chapter-level paging with forward/back navigation arrows
- Section numbers in the margin
- "Focus" and "cross-reference" panels (Greek ↔ English side by side)
- The Perseus 4 color scheme, nav bars, and typography
- Book/chapter browse bar
- Toggle between Greek-focus and English-focus views

**Approach:** Maximum HTML5 + CSS, minimal JavaScript (only for the show/hide toggle of the secondary text and browse bar).

## 1. Configuration

In [1]:
import os
from pathlib import Path

# ── Input files ──────────────────────────────────────────────
GREEK_XML  = "/Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-grc2.xml"
ENG_XML    = "/Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-eng2.xml"

# ── Output directory ─────────────────────────────────────────
OUT_DIR    = Path("/Users/gcrane/Downloads/claude/perseus_site")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Greek XML:   {GREEK_XML}")
print(f"English XML: {ENG_XML}")
print(f"Output dir:  {OUT_DIR}")

Greek XML:   /Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-grc2.xml
English XML: /Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-eng2.xml
Output dir:  /Users/gcrane/Downloads/claude/perseus_site


## 2. Parse the TEI XML

In [2]:
import xml.etree.ElementTree as ET
import re
from collections import OrderedDict

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

def extract_text_recursive(elem):
    """Extract the readable text from a TEI element, stripping markup but
    keeping <note> content in parentheses and <placeName>/<name> text inline."""
    parts = []
    if elem.text:
        parts.append(elem.text)
    for child in elem:
        tag = child.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
        if tag == 'note':
            # Render editorial notes as small footnote-style text
            note_text = extract_text_recursive(child).strip()
            if note_text:
                parts.append(f'<span class="note">[{note_text}]</span>')
        elif tag == 'milestone':
            # paragraph milestone → nothing visible
            pass
        elif tag == 'reg':
            # skip the regularized geo forms inside <name>
            pass
        elif tag == 'placeName':
            parts.append(extract_text_recursive(child))
        else:
            parts.append(extract_text_recursive(child))
        if child.tail:
            parts.append(child.tail)
    return ''.join(parts)


def parse_tei(path):
    """Parse a TEI XML file into a nested dict:
       { book_n: { chapter_n: { section_n: text_html } } }
    Also returns metadata (title, author, editor).
    """
    tree = ET.parse(path)
    root = tree.getroot()

    # Metadata
    title  = (root.findtext('.//tei:titleStmt/tei:title', namespaces=NS) or '').strip()
    author = (root.findtext('.//tei:titleStmt/tei:author', namespaces=NS) or '').strip()
    editor_el = root.find('.//tei:titleStmt/tei:editor', NS)
    editor = (editor_el.text or '').strip() if editor_el is not None else ''

    body = root.find('.//tei:body', NS)
    top_div = body.find('tei:div', NS)  # the edition/translation wrapper

    data = OrderedDict()
    for book_div in top_div.findall('tei:div[@type="textpart"]', NS):
        book_n = book_div.get('n')
        data[book_n] = OrderedDict()
        for chap_div in book_div.findall('tei:div[@type="textpart"]', NS):
            chap_n = chap_div.get('n')
            data[book_n][chap_n] = OrderedDict()
            for sec_div in chap_div.findall('tei:div[@type="textpart"]', NS):
                sec_n = sec_div.get('n')
                # Get <p> content
                paragraphs = sec_div.findall('tei:p', NS)
                html_parts = []
                for p in paragraphs:
                    html_parts.append(extract_text_recursive(p).strip())
                data[book_n][chap_n][sec_n] = ' '.join(html_parts)

    meta = {'title': title, 'author': author, 'editor': editor}
    return data, meta


grc_data, grc_meta = parse_tei(GREEK_XML)
eng_data, eng_meta = parse_tei(ENG_XML)

print(f"Greek:   {grc_meta['title']} by {grc_meta['author']} (ed. {grc_meta['editor']})")
print(f"English: {eng_meta['title']} by {eng_meta['author']} (tr. {eng_meta['editor']})")
print(f"Books:   {list(grc_data.keys())}")
print(f"Book 1 chapters: {len(grc_data['1'])}")

Greek:   Ἱστορίαι by Herodotus (ed. A.D. Godley)
English: The Histories by Herodotus (tr. A. D. Godley)
Books:   ['1', '2', '3', '4', '5', '6', '7', '8', '9']
Book 1 chapters: 216


## 3. Build the navigation index

For each chapter page we need to know: previous chapter, next chapter, what book we're in.

In [3]:
def build_chapter_list(data):
    """Return a flat list of (book_n, chapter_n) tuples in order."""
    chapters = []
    for book_n in data:
        for chap_n in data[book_n]:
            chapters.append((book_n, chap_n))
    return chapters

chapter_list = build_chapter_list(grc_data)
print(f"Total chapters across all books: {len(chapter_list)}")
print(f"First 10: {chapter_list[:10]}")

Total chapters across all books: 1578
First 10: [('1', '1'), ('1', '2'), ('1', '3'), ('1', '4'), ('1', '5'), ('1', '6'), ('1', '7'), ('1', '8'), ('1', '9'), ('1', '10')]


## 4. Define the Perseus 4 HTML/CSS Template

This is the core of the replication — the HTML structure and CSS that recreates
the Perseus 4 reading interface.

In [4]:
def page_filename(book, chapter, focus='greek'):
    """Generate the filename for a given book/chapter page."""
    return f"{focus}_book{book}_ch{chapter}.html"


PERSEUS_CSS = r"""
/* ================================================================
   Perseus 4.0 (Hopper) — Faithful CSS Recreation
   ================================================================ */

* { margin: 0; padding: 0; box-sizing: border-box; }

body {
    font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
    font-size: 14px;
    color: #333;
    background: #fff;
}

/* ── Top banner ─────────────────────────────────────────────── */
#perseus-banner {
    background: #6b3a2a;
    color: #fff;
    padding: 6px 16px;
    display: flex;
    align-items: center;
    justify-content: space-between;
}
#perseus-banner h1 {
    font-size: 18px;
    font-weight: normal;
    letter-spacing: 1px;
}
#perseus-banner h1 a { color: #fff; text-decoration: none; }
#perseus-banner .doc-title {
    font-size: 13px;
    color: #e8d8c8;
}

/* ── Grey nav bar (collections bar) ─────────────────────────── */
#nav-bar {
    background: #d6d6c8;
    border-bottom: 1px solid #b0b0a0;
    padding: 4px 16px;
    font-size: 12px;
    color: #555;
}
#nav-bar a {
    color: #336;
    text-decoration: none;
    margin-right: 12px;
}
#nav-bar a:hover { text-decoration: underline; }

/* ── Browse bar ─────────────────────────────────────────────── */
#browse-bar {
    background: #e8e8dc;
    border-bottom: 1px solid #ccc;
    padding: 6px 16px;
    font-size: 12px;
}
#browse-bar summary {
    cursor: pointer;
    color: #336;
    font-weight: bold;
    font-size: 12px;
}
#browse-bar .book-links,
#browse-bar .chapter-links {
    margin-top: 4px;
}
#browse-bar a {
    color: #336;
    text-decoration: none;
    margin-right: 6px;
    padding: 1px 4px;
}
#browse-bar a:hover { background: #d0d0c0; }
#browse-bar a.current {
    background: #6b3a2a;
    color: #fff;
    border-radius: 2px;
}
#browse-bar label {
    font-weight: bold;
    color: #555;
    margin-right: 4px;
}

/* ── Main content area ──────────────────────────────────────── */
#main-container {
    display: flex;
    max-width: 1100px;
    margin: 0 auto;
    padding: 16px;
    gap: 20px;
}

/* Focus text (left/primary column) */
#focus-text {
    flex: 3;
    min-width: 0;
}

/* Cross-reference text (right/secondary column) */
#cross-ref {
    flex: 2;
    min-width: 0;
    border-left: 1px solid #ccc;
    padding-left: 20px;
}
#cross-ref .panel-header {
    background: #e8e8dc;
    border: 1px solid #ccc;
    padding: 4px 10px;
    font-size: 12px;
    font-weight: bold;
    color: #555;
    margin-bottom: 10px;
}

/* ── Navigation arrows ──────────────────────────────────────── */
.nav-arrows {
    margin: 8px 0;
    display: flex;
    align-items: center;
    gap: 8px;
}
.nav-arrows a {
    color: #336;
    text-decoration: none;
    font-size: 20px;
    line-height: 1;
    padding: 2px 6px;
}
.nav-arrows a:hover {
    background: #e0e0d4;
}
.nav-arrows .location {
    font-size: 12px;
    color: #666;
}

/* ── Section text styling ───────────────────────────────────── */
.section-block {
    margin-bottom: 2px;
    position: relative;
    padding-left: 40px;
    line-height: 1.75;
}
.section-block .sec-num {
    position: absolute;
    left: 0;
    top: 0;
    color: #990000;
    font-size: 12px;
    font-weight: bold;
    width: 32px;
    text-align: right;
}
.section-block p {
    display: inline;
}

/* Greek text */
.greek-text {
    font-size: 16px;
    line-height: 1.8;
}

/* English text in secondary panel */
.english-text {
    font-size: 13px;
    line-height: 1.7;
    color: #444;
}

/* Notes */
.note {
    font-size: 11px;
    color: #888;
    font-style: italic;
}

/* ── Credits / footer ──────────────────────────────────────── */
#credits {
    background: #f0f0e8;
    border-top: 1px solid #ccc;
    padding: 10px 16px;
    font-size: 11px;
    color: #666;
    margin-top: 20px;
}

/* ── Version toggle ─────────────────────────────────────────── */
#version-select {
    background: #e8e8dc;
    border: 1px solid #ccc;
    padding: 4px 10px;
    font-size: 12px;
    margin-bottom: 12px;
}
#version-select a {
    color: #336;
    text-decoration: none;
    margin-right: 10px;
}
#version-select a:hover { text-decoration: underline; }
#version-select a.active {
    font-weight: bold;
    color: #333;
}

/* ── Responsive ─────────────────────────────────────────────── */
@media (max-width: 768px) {
    #main-container {
        flex-direction: column;
    }
    #cross-ref {
        border-left: none;
        border-top: 1px solid #ccc;
        padding-left: 0;
        padding-top: 16px;
    }
}
"""

print(f"CSS template: {len(PERSEUS_CSS)} chars")

CSS template: 4543 chars


## 5. HTML Page Generator

In [5]:
from html import escape

BOOK_NAMES = {
    '1': 'Clio', '2': 'Euterpe', '3': 'Thalia', '4': 'Melpomene',
    '5': 'Terpsichore', '6': 'Erato', '7': 'Polymnia',
    '8': 'Urania', '9': 'Calliope'
}


def render_sections(sections_dict, css_class='greek-text'):
    """Render a chapter's sections as HTML with marginal section numbers."""
    html = []
    for sec_n, text in sections_dict.items():
        label = sec_n if sec_n not in ('0', 'pr') else 'pr'
        html.append(
            f'<div class="section-block {css_class}">'
            f'<span class="sec-num">[{label}]</span>'
            f'<p>{text}</p>'
            f'</div>'
        )
    return '\n'.join(html)


def render_browse_bar(book_n, chap_n, focus, data):
    """Render the book/chapter browse bar."""
    # Book links
    book_links = []
    for bn in data:
        first_ch = list(data[bn].keys())[0]
        cls = ' class="current"' if bn == book_n else ''
        muse = f' ({BOOK_NAMES.get(bn, "")})' if bn in BOOK_NAMES else ''
        book_links.append(
            f'<a href="{page_filename(bn, first_ch, focus)}"{cls}>Book {bn}{muse}</a>'
        )
    # Chapter links for current book
    chap_links = []
    for cn in data[book_n]:
        cls = ' class="current"' if cn == chap_n else ''
        chap_links.append(
            f'<a href="{page_filename(book_n, cn, focus)}"{cls}>{cn}</a>'
        )

    return f"""<div id="browse-bar">
  <details open>
    <summary>Browse: Herodotus, Histories</summary>
    <div class="book-links">
      <label>Book:</label> {' '.join(book_links)}
    </div>
    <div class="chapter-links">
      <label>Chapter:</label> {' '.join(chap_links)}
    </div>
  </details>
</div>"""


def generate_page(book_n, chap_n, focus, chapter_list, grc_data, eng_data):
    """Generate a complete Perseus 4-style HTML page for one chapter."""
    idx = chapter_list.index((book_n, chap_n))
    prev_link = ''
    next_link = ''
    if idx > 0:
        pb, pc = chapter_list[idx - 1]
        prev_link = f'<a href="{page_filename(pb, pc, focus)}" title="Previous">&#x25C0;</a>'
    if idx < len(chapter_list) - 1:
        nb, nc = chapter_list[idx + 1]
        next_link = f'<a href="{page_filename(nb, nc, focus)}" title="Next">&#x25B6;</a>'

    location_str = f"Hdt. {book_n}.{chap_n}"
    nav_html = f"""<div class="nav-arrows">
  {prev_link}
  <span class="location">{location_str}</span>
  {next_link}
</div>"""

    # Determine which text is focus, which is cross-ref
    if focus == 'greek':
        focus_data = grc_data
        cross_data = eng_data
        focus_class = 'greek-text'
        cross_class = 'english-text'
        focus_label = f"Herodotus, {grc_meta['title']}"
        cross_label = f"English (tr. {eng_meta['editor']})"
        other_focus = 'english'
        focus_ed = f"ed. {grc_meta['editor']}"
    else:
        focus_data = eng_data
        cross_data = grc_data
        focus_class = 'english-text'
        cross_class = 'greek-text'
        focus_label = f"Herodotus, {eng_meta['title']} (tr. {eng_meta['editor']})"
        cross_label = f"Greek ({grc_meta['title']})"
        other_focus = 'greek'
        focus_ed = f"tr. {eng_meta['editor']}"

    # Get sections
    focus_sections = focus_data.get(book_n, {}).get(chap_n, {})
    cross_sections = cross_data.get(book_n, {}).get(chap_n, {})

    focus_html = render_sections(focus_sections, focus_class)
    cross_html = render_sections(cross_sections, cross_class)

    browse_bar = render_browse_bar(book_n, chap_n, focus, focus_data)

    muse = BOOK_NAMES.get(book_n, '')
    book_display = f"Book {book_n}" + (f" ({muse})" if muse else '')

    page = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Herodotus, Histories, {book_display}, chapter {chap_n} — Perseus Digital Library</title>
  <style>
{PERSEUS_CSS}
  </style>
</head>
<body>

<!-- ═══ BANNER ═══ -->
<div id="perseus-banner">
  <h1><a href="index.html">Perseus Digital Library</a></h1>
  <span class="doc-title">{focus_label}</span>
</div>

<!-- ═══ NAV BAR ═══ -->
<div id="nav-bar">
  <a href="index.html">Home</a>
  <a href="{page_filename(book_n, chap_n, other_focus)}">Switch to {other_focus.title()} focus</a>
</div>

<!-- ═══ BROWSE BAR ═══ -->
{browse_bar}

<!-- ═══ MAIN CONTENT ═══ -->
<div id="main-container">

  <!-- Focus Text -->
  <div id="focus-text">
    <div id="version-select">
      <a href="{page_filename(book_n, chap_n, 'greek')}"
         class="{'active' if focus=='greek' else ''}">Greek (Godley)</a>
      <a href="{page_filename(book_n, chap_n, 'english')}"
         class="{'active' if focus=='english' else ''}">English (Godley)</a>
    </div>

    {nav_html}

    <h2 style="font-size:14px; color:#555; margin-bottom:12px;">
      {book_display}, chapter {chap_n}
    </h2>

    {focus_html}

    {nav_html}

    <div id="credits">
      <p><strong>Herodotus.</strong> <em>Herodotus</em>, {focus_ed}.
         Cambridge: Harvard University Press; London: William Heinemann Ltd. 1920–1925.</p>
      <p style="margin-top:4px;">Provided by the Perseus Digital Library.
         Original text under Creative Commons ShareAlike 3.0 License.</p>
    </div>
  </div>

  <!-- Cross-reference Text -->
  <div id="cross-ref">
    <div class="panel-header">{cross_label}</div>
    {cross_html}
  </div>

</div>

</body>
</html>"""
    return page

print("Page generator defined.")

Page generator defined.


## 6. Generate the Index Page

In [6]:
def generate_index(grc_data):
    """Generate a table-of-contents index page."""
    toc_items = []
    for book_n in grc_data:
        muse = BOOK_NAMES.get(book_n, '')
        first_ch = list(grc_data[book_n].keys())[0]
        toc_items.append(
            f'<li style="margin-bottom:10px;">'
            f'<a href="{page_filename(book_n, first_ch, "greek")}">'
            f'<strong>Book {book_n}</strong> — {muse}</a>'
            f' <span style="color:#888; font-size:12px;">'
            f'({len(grc_data[book_n])} chapters)</span></li>'
        )

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Herodotus, Histories — Perseus Digital Library</title>
  <style>
{PERSEUS_CSS}
  </style>
</head>
<body>

<div id="perseus-banner">
  <h1><a href="index.html">Perseus Digital Library</a></h1>
  <span class="doc-title">Herodotus, Histories</span>
</div>

<div id="nav-bar">
  <a href="index.html">Home</a>
  <span>Collections &raquo; Greek and Roman Materials &raquo; Herodotus</span>
</div>

<div id="main-container">
  <div id="focus-text" style="flex:1; max-width:700px;">
    <h2 style="font-size:20px; color:#6b3a2a; margin-bottom:6px;">Herodotus, Histories</h2>
    <p style="font-size:13px; color:#666; margin-bottom:16px;">
      Greek text: ed. A. D. Godley<br>
      English translation: tr. A. D. Godley<br>
      Cambridge: Harvard University Press, 1920–1925.
    </p>

    <div id="version-select">
      <strong>Choose a focus text:</strong>
      <a href="{page_filename('1', list(grc_data['1'].keys())[0], 'greek')}">Greek</a>
      <a href="{page_filename('1', list(grc_data['1'].keys())[0], 'english')}">English Translation</a>
    </div>

    <h3 style="font-size:14px; color:#555; margin: 16px 0 8px;">Table of Contents</h3>
    <ol style="padding-left:20px; line-height:1.8;">
      {''.join(toc_items)}
    </ol>
  </div>
</div>

</body>
</html>"""

print("Index generator defined.")

Index generator defined.


## 7. Generate All Pages

This iterates through every book/chapter and writes two HTML files per chapter
(one Greek-focus, one English-focus), plus the index.

In [7]:
import time

start = time.time()

# Write index
index_html = generate_index(grc_data)
(OUT_DIR / 'index.html').write_text(index_html, encoding='utf-8')

# Write chapter pages
page_count = 0
for book_n in grc_data:
    for chap_n in grc_data[book_n]:
        for focus in ('greek', 'english'):
            html = generate_page(book_n, chap_n, focus,
                                 chapter_list, grc_data, eng_data)
            fname = page_filename(book_n, chap_n, focus)
            (OUT_DIR / fname).write_text(html, encoding='utf-8')
            page_count += 1

elapsed = time.time() - start
print(f"Generated {page_count} chapter pages + 1 index in {elapsed:.1f}s")
print(f"Output: {OUT_DIR}")

Generated 3156 chapter pages + 1 index in 1.2s
Output: /Users/gcrane/Downloads/claude/perseus_site


## 8. Quick Verification

Let's look at the generated output for Book 1, Chapter 1 to make sure it looks right.

In [8]:
# Show a snippet of the generated page
sample = (OUT_DIR / 'greek_book1_ch1.html').read_text(encoding='utf-8')
# Show first 3000 chars
print(sample[:3000])
print('\n... [truncated] ...')
print(f'\nTotal page size: {len(sample):,} bytes')

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Herodotus, Histories, Book 1 (Clio), chapter 1 — Perseus Digital Library</title>
  <style>

/* ================================================================
   Perseus 4.0 (Hopper) — Faithful CSS Recreation
   ================================================================ */

* { margin: 0; padding: 0; box-sizing: border-box; }

body {
    font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
    font-size: 14px;
    color: #333;
    background: #fff;
}

/* ── Top banner ─────────────────────────────────────────────── */
#perseus-banner {
    background: #6b3a2a;
    color: #fff;
    padding: 6px 16px;
    display: flex;
    align-items: center;
    justify-content: space-between;
}
#perseus-banner h1 {
    font-size: 18px;
    font-weight: normal;
    letter-spacing: 1px;
}
#perseus-banner h1 a { color: #fff; text

In [9]:
# Summary stats
import os

total_size = 0
file_count = 0
for f in OUT_DIR.iterdir():
    if f.suffix == '.html':
        total_size += f.stat().st_size
        file_count += 1

print(f"Total HTML files: {file_count}")
print(f"Total size:       {total_size / 1024 / 1024:.1f} MB")
print(f"Average page:     {total_size / file_count / 1024:.1f} KB")

Total HTML files: 3157
Total size:       56.5 MB
Average page:     18.3 KB


## 9. Preview in Notebook

Render Book 1, Chapter 1 (Greek focus) inline to see the Perseus 4 look.

In [10]:
from IPython.display import IFrame, display, HTML

# Render inline preview
preview_html = (OUT_DIR / 'greek_book1_ch1.html').read_text(encoding='utf-8')
display(HTML(f'<iframe srcdoc="{preview_html.replace(chr(34), "&quot;")}" '
             f'width="100%" height="700" style="border:1px solid #ccc;"></iframe>'))

/Users/gcrane/Library/Python/3.8/lib/python/site-packages/IPython/core/display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 10. Package for Deployment

Create a zip file of the entire site, ready to be served from any static host.

In [12]:
import shutil

zip_path = '//Users/gcrane/Downloads/perseus_herodotus_site'
shutil.make_archive(zip_path, 'zip', OUT_DIR)
zip_size = os.path.getsize(zip_path + '.zip') / 1024 / 1024
print(f"Site archive: {zip_path}.zip ({zip_size:.1f} MB)")
print(f"\nTo serve locally:")
print(f"  cd {OUT_DIR} && python3 -m http.server 8000")
print(f"  Open http://localhost:8000/index.html")

Site archive: //Users/gcrane/Downloads/perseus_herodotus_site.zip (14.5 MB)

To serve locally:
  cd /Users/gcrane/Downloads/claude/perseus_site && python3 -m http.server 8000
  Open http://localhost:8000/index.html


---

## Design Notes

### What this replicates from Perseus 4:

1. **The maroon banner** with "Perseus Digital Library" branding
2. **The grey collections nav bar** beneath it
3. **The collapsible browse bar** with book/chapter links (using `<details>` — pure HTML5, no JS)
4. **Blue navigation arrows** (◀ ▶) for paging through chapters
5. **Section numbers in the left margin** in red, matching P4's `[1]`, `[2]`… style
6. **Two-column layout**: focus text (larger) on the left, cross-reference (translation or Greek) on the right
7. **Version selector** to switch between Greek-focus and English-focus views
8. **Credits block** at the bottom of each page
9. **Palatino/Georgia serif typography** matching P4's text rendering

### What's intentionally omitted (for now):
- Morphological analysis / word study tool
- Dictionary lookup sidebar
- Full-text search
- Commentaries
- Named entity linking
- The P4 "text identifier" jump box

### JavaScript used:
- **None.** The browse bar toggle uses `<details>/<summary>` (HTML5).
  The Greek/English switching uses pre-generated paired HTML files.
  Navigation arrows are plain `<a>` links.